# Stage 2: check each helper in a small run

Each helper script is exercised here on a handful of images before any training loop depends on it. Look at the pictures and read the PASS lines. Nothing here trains a model for more than a moment.

In [ ]:
# @title Find the project code (mounts Drive on a fresh Colab machine)
import os
CODE_DIRS = ["/content/mnist", "/content/drive/MyDrive/colab-mnist-test"]  # uploaded by the Colab tool, or your Drive folder
if os.path.isdir("/content") and not any(os.path.isfile(os.path.join(d, "models.py")) for d in CODE_DIRS):
    from google.colab import drive
    drive.mount("/content/drive")  # a pop-up asks you to approve: click Connect to Google Drive within two minutes
found = [d for d in CODE_DIRS if os.path.isfile(os.path.join(d, "models.py"))]
print("project code found in:", found or "nowhere on this machine (this computer: the folder above the notebook)")

In [ ]:
# @title Setup
import json, os, sys, time
for path in ("..", *CODE_DIRS):  # project root on this computer, uploaded or Drive folder on Colab
    if os.path.isfile(os.path.join(path, "models.py")) and os.path.abspath(path) not in sys.path:
        sys.path.insert(0, os.path.abspath(path))
import matplotlib.pyplot as plt
import torch
import augment, checks, data, models, quicktrain, trainer

ON_COLAB = os.path.isdir("/content")
DATA_DIR = "/content/data" if ON_COLAB else "../data"
FIG_DIR = "/content/figures" if ON_COLAB else "figures"
os.makedirs(FIG_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 0
print("device:", device, "| torch", torch.__version__)

In [ ]:
# @title Helper 1: loading and splitting the data
from torchvision import datasets
raw = datasets.MNIST(DATA_DIR, train=True, download=True)           # plain images, for display
train_full, val_source, test_set = data.load_mnist(DATA_DIR, augment_train=True)
train_set, val_set, train_idx, val_idx = data.make_sets(train_full, val_source, 2000, 500, SEED)
print("training items:", len(train_set), "| validation items:", len(val_set), "| test items:", len(test_set))
results = [checks.check_no_overlap(train_idx, val_idx)]
print(checks.format_results(results))

In [ ]:
# @title Helper 2: augmentation, the original beside three random versions
plain, augmented = augment.build_transform(False), augment.build_transform(True)
fig, axes = plt.subplots(4, 6, figsize=(9, 6.2))
for row, i in enumerate(train_idx[:6]):
    image, label = raw[i]
    for col, transform in enumerate([plain, augmented, augmented, augmented]):
        axes[col, row].imshow(transform(image)[0], cmap="gray")
        axes[col, row].axis("off")
    axes[0, row].set_title(f"digit {label}", fontsize=10)
axes[0, 0].text(-6, 14, "original", rotation=90, va="center", fontsize=9)
for r in (1, 2, 3):
    axes[r, 0].text(-6, 14, "augmented", rotation=90, va="center", fontsize=9)
fig.suptitle("Augmentation should move and tilt a digit slightly without changing which digit it is")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "helpers_augmentation.png"), dpi=110); plt.show()

In [ ]:
# @title Helper 3: class balance of the training subset
from collections import Counter
counts = Counter(raw.targets[i].item() for i in train_idx)
plt.figure(figsize=(6, 2.6)); plt.bar(sorted(counts), [counts[k] for k in sorted(counts)], color="#b56a00")
plt.xticks(range(10)); plt.xlabel("digit"); plt.ylabel("images"); plt.title("Images per digit")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "helpers_class_counts.png"), dpi=110); plt.show()
print("smallest class:", min(counts.values()), "images | largest class:", max(counts.values()), "images")

In [ ]:
# @title Helper 4: every model builds and gives ten scores per image
import models_extra
from torch.utils.data import DataLoader
x, y = next(iter(DataLoader(train_set, batch_size=32)))
all_models = {name: models.build_model(name) for name in models.MODEL_NAMES}
all_models["wide_mlp"] = models_extra.WideMLP()
for name, model in all_models.items():
    out = model(x)
    print(f"{name:9s} parameters {sum(p.numel() for p in model.parameters()):>9,}  output {tuple(out.shape)}")
    assert out.shape == (32, 10)
print("batch of images:", tuple(x.shape), "| labels:", y[:8].tolist(), "...")

In [ ]:
# @title Helper 5: one training step for each model, timed
for name, model in all_models.items():
    model = model.to(device)
    started = time.time()
    losses = quicktrain.train_steps(model, DataLoader(train_set, batch_size=64, shuffle=True), 5, device)
    print(f"{name:9s} 5 steps in {time.time() - started:.2f} s, loss {losses[0]:.3f} -> {losses[-1]:.3f}")

When the pictures look right and every line above is PASS or a sensible number, move on to the training pipeline notebook (stages 3 and 4).